# Step X: ステップ名

**論文での該当箇所:** Section X.X.X, 式 (N)

**目的:** このステップで何を実装するかを1-2行で書く

**参考:** [disassemble-channel - Transformerをゼロから実装する](https://disassemble-channel.com/transformer-from-scratch/)

---

このノートブックは `00_template.ipynb` のコピー。新しい Step に進むときは、これを複製して `0X_step_name.ipynb` にリネームしてから写経を始める。

## 1. 数式 (論文より)

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V
$$

(↑ 例: 該当 Step の数式を LaTeX で書く)

## 2. import

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# 再現性のため seed 固定
torch.manual_seed(0)

## 3. 実装(写経)

記事のコードを写経する。**コピペではなく自分で打つこと**(タイプすることで体に入る)。
分からない箇所はコメントで `# TODO: 後で理解` と残して進める。

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Scaled Dot-Product Attention.

    Args:
        Q: (batch, seq_q, d_k)
        K: (batch, seq_k, d_k)
        V: (batch, seq_k, d_v)
        mask: 適用するマスク(省略可)

    Returns:
        output: (batch, seq_q, d_v)
        attn_weights: (batch, seq_q, seq_k)
    """
    d_k = Q.size(-1)
    # QK^T を計算してスケーリング
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    # mask があれば適用(softmax 前に -1e9 を入れる)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    # softmax で重みに変換
    attn_weights = torch.softmax(scores, dim=-1)
    # 重みで V を加重平均
    output = torch.matmul(attn_weights, V)
    return output, attn_weights

## 4. 動作確認(shape チェック)

ダミーテンソルを作って、入出力の shape が期待通りか確認する。
**この `assert` がこの notebook のテスト**として機能する。

In [ ]:
# ダミー入力
batch, seq_len, d_k = 2, 5, 64
Q = torch.randn(batch, seq_len, d_k)
K = torch.randn(batch, seq_len, d_k)
V = torch.randn(batch, seq_len, d_k)

# 実行
output, attn = scaled_dot_product_attention(Q, K, V)

# shape チェック
assert output.shape == (batch, seq_len, d_k), f"想定 {(batch, seq_len, d_k)}, 実際 {output.shape}"
assert attn.shape == (batch, seq_len, seq_len), f"想定 {(batch, seq_len, seq_len)}, 実際 {attn.shape}"

# attention 重みが確率分布になっているか(各行が1に近い)
assert torch.allclose(attn.sum(dim=-1), torch.ones(batch, seq_len), atol=1e-5)

print('OK')
print('output:', output.shape)
print('attn:', attn.shape)

## 5. 可視化(任意)

Attention の場合: 重み行列をヒートマップで表示すると理解が深まる。

In [ ]:
import matplotlib.pyplot as plt

# 1サンプル目の attention 重みを可視化
plt.figure(figsize=(5, 4))
plt.imshow(attn[0].detach().numpy(), cmap='viridis')
plt.xlabel('Key position')
plt.ylabel('Query position')
plt.title('Attention weights (sample 0)')
plt.colorbar()
plt.show()

## 6. 理解メモ

写経して理解したポイントを箇条書きで残す。後で `notes/formula_to_code.md` に転記する材料になる。

- `√d_k` で割る理由: 内積が大きくなると softmax の勾配が消失するため
- `K.transpose(-2, -1)` は最後の2次元の入れ替え (= K^T)
- mask は softmax 前に `-1e9` を入れる(乗算ではない)
- (自分で気づいたポイントを足す)

## 7. 次のアクション

- [ ] このコードを `src/attention.py` に整理して移植
- [ ] `notes/formula_to_code.md` の該当セクションを更新
- [ ] 詰まった点があれば `notes/stuck_points.md` に記録
- [ ] `notes/learning_log.md` に今日の3行を書く
- [ ] `git add` + `git commit`